<a href="https://colab.research.google.com/github/shivangisinha828-beep/BeautyScout-/blob/main/BEAUTYSCOUT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install textblob beautifulsoup4 requests pandas numpy scikit-learn matplotlib seaborn wordcloud lxml html5lib

# For advanced scraping (if needed)
!pip install selenium webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import time
import random
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Option B: Create starter dataset (expand this with real data)
product_data = {
    'product_id': range(1, 31),
    'product_name': [
        # Moisturizers
        'CeraVe Moisturizing Cream', 'Neutrogena Hydro Boost Water Gel', 'Cetaphil DAM',
        'Kiehl\'s Ultra Facial Cream', 'Clinique Moisture Surge', 'The Ordinary Natural Moisturizing Factors',
        # Cleansers
        'CeraVe Hydrating Cleanser', 'Simple Kind to Skin Refreshing Wash', 'Cetaphil Gentle Cleanser',
        'La Roche-Posay Effaclar Cleanser', 'Plum Green Tea Pore Cleanser', 'Bioderma Sensibio H2O',
        # Serums
        'The Ordinary Niacinamide 10%', 'Minimalist 2% Salicylic Acid', 'Plum 15% Vitamin C',
        'L\'Oreal Paris 1.5% Hyaluronic Acid', 'The Ordinary Hyaluronic Acid 2%', 'Kiehl\'s Midnight Recovery',
        # Sunscreens
        'Neutrogena Ultra Sheer Dry Touch', 'La Shield Fisico SPF 50', 'Re\'equil Ultra Matte Dry Touch',
        'Aqualogica Glow+ Dewy Sunscreen', 'Minimalist SPF 50', 'Dr. Sheth\'s Haldi & Hyaluronic Sunscreen',
        # Haircare
        'L\'Oreal Paris Extraordinary Oil', 'Matrix Biolage Smoothproof', 'Plum Olive & Macadamia Hair Mask',
        'Mamaearth Onion Hair Oil', 'WOW Skin Science Red Onion Oil', 'Indulekha Bringha Hair Oil'
    ],
    'brand': [
        'CeraVe', 'Neutrogena', 'Cetaphil', 'Kiehl\'s', 'Clinique', 'The Ordinary',
        'CeraVe', 'Simple', 'Cetaphil', 'La Roche-Posay', 'Plum', 'Bioderma',
        'The Ordinary', 'Minimalist', 'Plum', 'L\'Oreal', 'The Ordinary', 'Kiehl\'s',
        'Neutrogena', 'La Shield', 'Re\'equil', 'Aqualogica', 'Minimalist', 'Dr. Sheth\'s',
        'L\'Oreal', 'Matrix', 'Plum', 'Mamaearth', 'WOW', 'Indulekha'
    ],
    'category': [
        'moisturizer', 'moisturizer', 'moisturizer', 'moisturizer', 'moisturizer', 'moisturizer',
        'cleanser', 'cleanser', 'cleanser', 'cleanser', 'cleanser', 'cleanser',
        'serum', 'serum', 'serum', 'serum', 'serum', 'serum',
        'sunscreen', 'sunscreen', 'sunscreen', 'sunscreen', 'sunscreen', 'sunscreen',
        'haircare', 'haircare', 'haircare', 'haircare', 'haircare', 'haircare'
    ],
    'skin_type': [
        'dry', 'combination', 'dry', 'all', 'all', 'all',
        'dry', 'all', 'all', 'oily', 'oily', 'all',
        'all', 'oily', 'all', 'all', 'all', 'all',
        'all', 'all', 'oily', 'all', 'all', 'all',
        'all', 'all', 'all', 'all', 'all', 'all'
    ],
    'concerns': [
        'hydration', 'hydration', 'dryness', 'hydration', 'hydration', 'hydration',
        'dryness', 'sensitive', 'sensitive', 'acne', 'acne', 'sensitive',
        'acne,brightening', 'acne,exfoliation', 'brightening', 'hydration', 'hydration', 'aging',
        'sun protection', 'sun protection', 'sun protection,oil control', 'hydration', 'sun protection', 'brightening',
        'damage repair', 'frizz', 'damage', 'hair fall', 'hair fall', 'hair fall'
    ],
    'price': [
        1450, 550, 450, 2950, 2800, 750,
        1250, 250, 450, 1250, 450, 650,
        550, 550, 650, 550, 550, 4200,
        550, 650, 550, 399, 699, 499,
        450, 650, 550, 399, 399, 450
    ],
    'rating': [4.6, 4.4, 4.5, 4.5, 4.4, 4.3, 4.7, 4.2, 4.5, 4.4, 4.3, 4.5, 4.6, 4.4, 4.3, 4.4, 4.5, 4.3, 4.4, 4.3, 4.5, 4.2, 4.3, 4.4, 4.2, 4.3, 4.2, 4.1, 4.2, 4.0],
    'url_nykaa': [f"https://www.nykaa.com/product-{i}" for i in range(1, 31)],
    'url_purplle': [f"https://purplle.com/product-{i}" for i in range(1, 31)],
    'url_amazon': [f"https://amazon.in/dp/product{i}" for i in range(1, 31)]
}

df_products = pd.DataFrame(product_data)
print(f"Loaded {len(df_products)} products")
print(df_products.head())

Loaded 30 products
   product_id                      product_name       brand     category  \
0           1         CeraVe Moisturizing Cream      CeraVe  moisturizer   
1           2  Neutrogena Hydro Boost Water Gel  Neutrogena  moisturizer   
2           3                      Cetaphil DAM    Cetaphil  moisturizer   
3           4        Kiehl's Ultra Facial Cream     Kiehl's  moisturizer   
4           5           Clinique Moisture Surge    Clinique  moisturizer   

     skin_type   concerns  price  rating                        url_nykaa  \
0          dry  hydration   1450     4.6  https://www.nykaa.com/product-1   
1  combination  hydration    550     4.4  https://www.nykaa.com/product-2   
2          dry    dryness    450     4.5  https://www.nykaa.com/product-3   
3          all  hydration   2950     4.5  https://www.nykaa.com/product-4   
4          all  hydration   2800     4.4  https://www.nykaa.com/product-5   

                     url_purplle                     url_amaz

In [4]:
def get_user_preferences():
    print("=" * 50)
    print("🌸 BEAUTY SCOUT PRO - Find Your Perfect Match 🌸")
    print("=" * 50)

    # Category selection
    print("\n📋 PRODUCT CATEGORY:")
    print("1. Moisturizer\n2. Cleanser\n3. Serum\n4. Sunscreen\n5. Haircare")
    category_choice = input("Enter choice (1-5): ")
    category_map = {'1': 'moisturizer', '2': 'cleanser', '3': 'serum',
                    '4': 'sunscreen', '5': 'haircare'}
    user_category = category_map.get(category_choice, 'moisturizer')

    # Skin type
    print("\n💆‍♀️ SKIN TYPE:")
    print("1. Dry\n2. Oily\n3. Combination\n4. Normal/All")
    skin_choice = input("Enter choice (1-4): ")
    skin_map = {'1': 'dry', '2': 'oily', '3': 'combination', '4': 'all'}
    user_skin_type = skin_map.get(skin_choice, 'all')

    # Concerns
    print("\n🎯 SKIN CONCERNS (comma-separated):")
    print("Options: acne, hydration, brightening, aging, dryness, sensitive, oil_control")
    user_concerns = input("Enter concerns: ").lower().split(',')
    user_concerns = [c.strip() for c in user_concerns]

    # Budget
    print("\n💰 BUDGET RANGE (in INR):")
    min_budget = int(input("Minimum budget: ") or 0)
    max_budget = int(input("Maximum budget: ") or 5000)

    return {
        'category': user_category,
        'skin_type': user_skin_type,
        'concerns': user_concerns,
        'min_budget': min_budget,
        'max_budget': max_budget
    }

In [5]:
def build_product_vectors(df):
    """Create feature vectors for products based on attributes"""

    # Create text representation for each product
    product_texts = []
    for _, row in df.iterrows():
        text = f"{row['brand']} {row['category']} {row['skin_type']} {row['concerns']}"
        product_texts.append(text)

    # Convert to TF-IDF vectors
    vectorizer = TfidfVectorizer()
    product_vectors = vectorizer.fit_transform(product_texts)

    return product_vectors, vectorizer

def get_recommendations(user_prefs, df, product_vectors, vectorizer, top_n=5):
    """Recommend products based on user preferences"""

    # Filter by category and budget first
    filtered_df = df[
        (df['category'] == user_prefs['category']) &
        (df['price'] >= user_prefs['min_budget']) &
        (df['price'] <= user_prefs['max_budget'])
    ].copy()

    if len(filtered_df) == 0:
        print("No products match your criteria. Try expanding your budget!")
        return pd.DataFrame()

    # Create user preference vector
    user_text = f"{user_prefs['category']} {user_prefs['skin_type']} {' '.join(user_prefs['concerns'])}"
    user_vector = vectorizer.transform([user_text])

    # Get indices of filtered products
    filtered_indices = filtered_df.index.tolist()
    filtered_vectors = product_vectors[filtered_indices]

    # Calculate cosine similarity
    similarities = cosine_similarity(user_vector, filtered_vectors).flatten()

    # Add scores to dataframe
    filtered_df = filtered_df.copy()
    filtered_df['similarity_score'] = similarities

    # Sort by similarity and get top N
    recommendations = filtered_df.nlargest(top_n, 'similarity_score')

    # Add final score (combination of rating + similarity)
    recommendations['final_score'] = (
        recommendations['similarity_score'] * 0.6 +
        (recommendations['rating'] / 5) * 0.4
    ) * 10  # Scale to 1-10

    return recommendations[['product_name', 'brand', 'price', 'rating',
                            'final_score', 'similarity_score', 'url_nykaa',
                            'url_purplle', 'url_amazon']]

In [6]:
def scrape_nykaa_reviews(product_url, max_reviews=5):
    """Scrape reviews from Nykaa product page"""
    # Note: You'll need to inspect Nykaa's actual HTML structure
    # This is a template that works with the pattern
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        response = requests.get(product_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # These selectors may need adjustment based on actual page structure
        reviews = []
        review_elements = soup.find_all('div', class_='review-text')[:max_reviews]

        for elem in review_elements:
            review_text = elem.get_text(strip=True)
            if review_text:
                # Analyze sentiment
                blob = TextBlob(review_text)
                sentiment = blob.sentiment.polarity
                reviews.append({
                    'platform': 'Nykaa',
                    'text': review_text[:500],  # Truncate
                    'sentiment': sentiment,
                    'rating': None  # Would extract rating separately
                })

        return reviews
    except Exception as e:
        print(f"Error scraping Nykaa: {e}")
        return []

def scrape_purplle_reviews(product_url, max_reviews=5):
    """Scrape reviews from Purplle"""
    # Similar structure - adapt based on Purplle's HTML
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        response = requests.get(product_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        reviews = []
        review_elements = soup.find_all('div', class_='review-desc')[:max_reviews]

        for elem in review_elements:
            review_text = elem.get_text(strip=True)
            if review_text:
                blob = TextBlob(review_text)
                sentiment = blob.sentiment.polarity
                reviews.append({
                    'platform': 'Purplle',
                    'text': review_text[:500],
                    'sentiment': sentiment,
                    'rating': None
                })

        return reviews
    except Exception as e:
        print(f"Error scraping Purplle: {e}")
        return []

def scrape_amazon_reviews(product_url, max_reviews=5):
    """Scrape reviews from Amazon India"""
    # Amazon's review pages have a specific structure
    # Convert product URL to reviews URL
    if '/dp/' in product_url:
        review_url = product_url.replace('/dp/', '/product-reviews/')
    else:
        review_url = product_url + "/product-reviews/"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        response = requests.get(review_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        reviews = []
        review_elements = soup.find_all('span', {'data-hook': 'review-body'})[:max_reviews]

        for elem in review_elements:
            review_text = elem.get_text(strip=True)
            if review_text:
                blob = TextBlob(review_text)
                sentiment = blob.sentiment.polarity
                reviews.append({
                    'platform': 'Amazon',
                    'text': review_text[:500],
                    'sentiment': sentiment,
                    'rating': None
                })

        return reviews
    except Exception as e:
        print(f"Error scraping Amazon: {e}")
        return []

def get_all_reviews_for_product(product_row, platforms=['nykaa', 'purplle', 'amazon']):
    """Aggregate reviews from multiple platforms for a product"""
    all_reviews = []

    if 'nykaa' in platforms and pd.notna(product_row.get('url_nykaa')):
        nykaa_reviews = scrape_nykaa_reviews(product_row['url_nykaa'])
        all_reviews.extend(nykaa_reviews)
        time.sleep(random.uniform(1, 2))  # Be polite

    if 'purplle' in platforms and pd.notna(product_row.get('url_purplle')):
        purplle_reviews = scrape_purplle_reviews(product_row['url_purplle'])
        all_reviews.extend(purplle_reviews)
        time.sleep(random.uniform(1, 2))

    if 'amazon' in platforms and pd.notna(product_row.get('url_amazon')):
        amazon_reviews = scrape_amazon_reviews(product_row['url_amazon'])
        all_reviews.extend(amazon_reviews)
        time.sleep(random.uniform(1, 2))

    return all_reviews

In [7]:
def find_optimum_review(reviews):
    """Find the most representative review (closest to average sentiment)"""
    if not reviews:
        return None

    # Calculate average sentiment across all reviews
    sentiments = [r['sentiment'] for r in reviews if r['sentiment'] is not None]
    if not sentiments:
        return None

    avg_sentiment = np.mean(sentiments)

    # Find review closest to average sentiment
    best_review = None
    best_distance = float('inf')

    for review in reviews:
        if review['sentiment'] is not None:
            distance = abs(review['sentiment'] - avg_sentiment)
            # Also prefer reviews with moderate length (50-300 chars)
            length_score = 1 if 50 <= len(review['text']) <= 300 else 0
            adjusted_distance = distance - (length_score * 0.05)

            if adjusted_distance < best_distance:
                best_distance = adjusted_distance
                best_review = review

    return best_review

def summarize_reviews(reviews):
    """Generate summary statistics and pros/cons from reviews"""
    if not reviews:
        return {"avg_sentiment": 0, "total_reviews": 0, "pros": [], "cons": []}

    sentiments = [r['sentiment'] for r in reviews if r['sentiment'] is not None]
    avg_sentiment = np.mean(sentiments) if sentiments else 0

    # Simple pros/cons extraction (can be enhanced with keyword matching)
    pro_keywords = ['love', 'great', 'amazing', 'perfect', 'best', 'good', 'nice', 'works']
    con_keywords = ['hate', 'bad', 'worst', 'terrible', 'disappointed', 'breakout', 'greasy']

    pros = set()
    cons = set()

    for review in reviews:
        text_lower = review['text'].lower()
        for word in pro_keywords:
            if word in text_lower:
                pros.add(word)
        for word in con_keywords:
            if word in text_lower:
                cons.add(word)

    return {
        'avg_sentiment': avg_sentiment,
        'total_reviews': len(reviews),
        'pros': list(pros)[:5],
        'cons': list(cons)[:5],
        'platforms': list(set([r['platform'] for r in reviews]))
    }

In [8]:
def main():
    # Step 1: Build product vectors
    print("🔄 Building product database...")
    product_vectors, vectorizer = build_product_vectors(df_products)

    # Step 2: Get user preferences
    user_prefs = get_user_preferences()

    # Step 3: Get recommendations
    print("\n🔍 Finding products that match your needs...")
    recommendations = get_recommendations(user_prefs, df_products, product_vectors, vectorizer, top_n=3)

    if len(recommendations) == 0:
        print("No products found. Try different preferences!")
        return

    # Step 4: For each recommendation, fetch real reviews
    print("\n📝 Fetching real reviews from Nykaa, Purplle, and Amazon...")
    print("   (This may take a moment)")
    print("-" * 60)

    for idx, (_, product) in enumerate(recommendations.iterrows(), 1):
        print(f"\n✨ RECOMMENDATION #{idx}: {product['product_name']}")
        print(f"   Brand: {product['brand']}")
        print(f"   Price: ₹{product['price']}")
        print(f"   Overall Score: {product['final_score']:.1f}/10")
        print(f"   Rating: {product['rating']}/5")
        print("-" * 40)

        # Fetch reviews
        reviews = get_all_reviews_for_product(product)

        if reviews:
            # Find optimum review
            optimum = find_optimum_review(reviews)
            summary = summarize_reviews(reviews)

            print(f"\n   📊 Review Summary:")
            print(f"      • Total reviews analyzed: {summary['total_reviews']}")
            print(f"      • Platforms: {', '.join(summary['platforms'])}")
            print(f"      • Average sentiment: {summary['avg_sentiment']:.2f} (from -1 to +1)")

            if summary['pros']:
                print(f"      • What users love: {', '.join(summary['pros'])}")
            if summary['cons']:
                print(f"      • Common concerns: {', '.join(summary['cons'])}")

            if optimum:
                print(f"\n   🌟 OPTIMUM REVIEW (Most representative):")
                print(f"      Platform: {optimum['platform']}")
                print(f"      \"{optimum['text'][:200]}...\"")
        else:
            print("\n   ⚠️ No reviews found. Try checking the product links directly!")

        print("\n   🔗 Where to buy:")
        if pd.notna(product.get('url_nykaa')):
            print(f"      • Nykaa: {product['url_nykaa']}")
        if pd.notna(product.get('url_purplle')):
            print(f"      • Purplle: {product['url_purplle']}")
        if pd.notna(product.get('url_amazon')):
            print(f"      • Amazon: {product['url_amazon']}")

        print("\n" + "=" * 60)

    # Step 5: Optional - Save recommendations to CSV
    save_choice = input("\n💾 Save these recommendations to CSV? (y/n): ")
    if save_choice.lower() == 'y':
        recommendations.to_csv('beauty_recommendations.csv', index=False)
        files.download('beauty_recommendations.csv')
        print("✅ Saved as 'beauty_recommendations.csv'")

# Run the app
if __name__ == "__main__":
    main()

🔄 Building product database...
🌸 BEAUTY SCOUT PRO - Find Your Perfect Match 🌸

📋 PRODUCT CATEGORY:
1. Moisturizer
2. Cleanser
3. Serum
4. Sunscreen
5. Haircare
Enter choice (1-5): 4

💆‍♀️ SKIN TYPE:
1. Dry
2. Oily
3. Combination
4. Normal/All
Enter choice (1-4): 1

🎯 SKIN CONCERNS (comma-separated):
Options: acne, hydration, brightening, aging, dryness, sensitive, oil_control
Enter concerns: acne

💰 BUDGET RANGE (in INR):
Minimum budget: 5000
Maximum budget: 7000

🔍 Finding products that match your needs...
No products match your criteria. Try expanding your budget!
No products found. Try different preferences!
